In [3]:
import pandas as pd
from sqlalchemy import create_engine

# Load dataset
df = pd.read_csv("netflix_titles.csv")
ott = df.copy()

# Handle missing values
ott["director"] = ott["director"].fillna("Unknown")
ott["cast"] = ott["cast"].fillna("Unknown")
ott["country"] = ott["country"].fillna("Unknown")
ott.dropna(subset=["date_added", "rating", "duration"], inplace=True)

# Fix date column
ott["date_added"] = pd.to_datetime(ott["date_added"].str.strip())
ott["year_added"] = ott["date_added"].dt.year
ott["month_added"] = ott["date_added"].dt.month_name()

# Split multi-value columns (genre, country) — take primary value
ott["primary_country"] = ott["country"].apply(lambda x: x.split(",")[0].strip())
ott["primary_genre"] = ott["listed_in"].apply(lambda x: x.split(",")[0].strip())

# Split duration into numeric fields (Movies = minutes, TV Shows = seasons)
ott["duration_value"] = ott["duration"].str.extract(r"(\d+)").astype(float)
ott["duration_unit"] = ott["duration"].apply(lambda x: "Season(s)" if "Season" in x else "min")

# Remove duplicates
ott.drop_duplicates(subset=["show_id"], inplace=True)

print("Original Shape:", df.shape)
print("Cleaned Shape:", ott.shape)
ott.head()

Original Shape: (8807, 12)
Cleaned Shape: (8790, 18)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,primary_country,primary_genre,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021,September,United States,Documentaries,90.0,min
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021,September,South Africa,International TV Shows,2.0,Season(s)
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021,September,Unknown,Crime TV Shows,1.0,Season(s)
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021,September,Unknown,Docuseries,1.0,Season(s)
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021,September,India,International TV Shows,2.0,Season(s)


In [4]:
!pip install sqlalchemy pyodbc

Defaulting to user installation because normal site-packages is not writeable


In [5]:
import pandas as pd
from sqlalchemy import create_engine

# Load dataset
df = pd.read_csv("netflix_titles.csv")
ott = df.copy()

# Handle missing values
ott["director"] = ott["director"].fillna("Unknown")
ott["cast"] = ott["cast"].fillna("Unknown")
ott["country"] = ott["country"].fillna("Unknown")
ott.dropna(subset=["date_added", "rating", "duration"], inplace=True)

# Fix date column
ott["date_added"] = pd.to_datetime(ott["date_added"].str.strip())
ott["year_added"] = ott["date_added"].dt.year
ott["month_added"] = ott["date_added"].dt.month_name()

# Split multi-value columns (genre, country) — take primary value
ott["primary_country"] = ott["country"].apply(lambda x: x.split(",")[0].strip())
ott["primary_genre"] = ott["listed_in"].apply(lambda x: x.split(",")[0].strip())

# Split duration into numeric fields (Movies = minutes, TV Shows = seasons)
ott["duration_value"] = ott["duration"].str.extract(r"(\d+)").astype(float)
ott["duration_unit"] = ott["duration"].apply(lambda x: "Season(s)" if "Season" in x else "min")

# Remove duplicates
ott.drop_duplicates(subset=["show_id"], inplace=True)

print("Original Shape:", df.shape)
print("Cleaned Shape:", ott.shape)
ott.head()

Original Shape: (8807, 12)
Cleaned Shape: (8790, 18)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,primary_country,primary_genre,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021,September,United States,Documentaries,90.0,min
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021,September,South Africa,International TV Shows,2.0,Season(s)
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021,September,Unknown,Crime TV Shows,1.0,Season(s)
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021,September,Unknown,Docuseries,1.0,Season(s)
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021,September,India,International TV Shows,2.0,Season(s)


In [6]:
server = "localhost\\SQLEXPRESS"
database = "OTT_Analytics"

conn_str = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
engine = create_engine(conn_str)

ott.to_sql(
    "Content_Library",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)
print("Loaded into SQL Server.")

OperationalError: (pyodbc.OperationalError) ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]SQL Server Network Interfaces: Error Locating Server/Instance Specified [xFFFFFFFF].  (-1) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 17 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to SQL Server. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (-1)')
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
server = "localhost"
database = "OTT_Analytics"

conn_str = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
engine = create_engine(conn_str)

In [ ]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=OTT_Analytics;"
    "Trusted_Connection=yes;"
)
print("Connected successfully!")
conn.close()

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# Load dataset
df = pd.read_csv("netflix_titles.csv")
ott = df.copy()

# Handle missing values
ott["director"] = ott["director"].fillna("Unknown")
ott["cast"] = ott["cast"].fillna("Unknown")
ott["country"] = ott["country"].fillna("Unknown")
ott.dropna(subset=["date_added", "rating", "duration"], inplace=True)

# Fix date column
ott["date_added"] = pd.to_datetime(ott["date_added"].str.strip())
ott["year_added"] = ott["date_added"].dt.year
ott["month_added"] = ott["date_added"].dt.month_name()

# Split multi-value columns
ott["primary_country"] = ott["country"].apply(lambda x: x.split(",")[0].strip())
ott["primary_genre"] = ott["listed_in"].apply(lambda x: x.split(",")[0].strip())

# Split duration
ott["duration_value"] = ott["duration"].str.extract(r"(\d+)").astype(float)
ott["duration_unit"] = ott["duration"].apply(lambda x: "Season(s)" if "Season" in x else "min")

ott.drop_duplicates(subset=["show_id"], inplace=True)

print("Cleaned Shape:", ott.shape)
ott.head()

In [ ]:
server = "localhost"
database = "OTT_Analytics"

conn_str = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
engine = create_engine(conn_str)

ott.to_sql(
    "Content_Library",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)
print("Loaded into SQL Server.")

In [ ]:
ott.to_csv("ott_cleaned.csv", index=False)
print("CSV exported successfully.")